# 导入依赖和定义文件

In [15]:

import json

import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [16]:

# 加载特征数据
X = np.load("X.npy")
X_test = np.load("X_test.npy")
y = np.load("y.npy")

# 划分训练集与验证集

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, train_size=0.8, test_size=0.2, random_state=0)


# 寻找并保存最优参数

In [17]:

# 定义 LGBM 的目标函数
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 6),
        'num_leaves': trial.suggest_int('num_leaves', 20, 30),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 定义 CatBoost 的目标函数
def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 6),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 3),
        'border_count': trial.suggest_categorical('border_count', [32, 64]),
        'verbose': False,
        'random_state': 0
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 创建 Optuna 研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=50)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=50)

# 获取最优参数
lgbm_best_params = lgbm_study.best_params
lgbm_best_value = lgbm_study.best_value
catboost_best_params = catboost_study.best_params
catboost_best_value = catboost_study.best_value

[I 2025-04-27 15:08:54,326] A new study created in memory with name: no-name-810137ae-3255-459c-971b-ce6b69bf06a3
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:54,368] Trial 0 finished with value: 0.7711328349626222 and parameters: {'n_estimators': 50, 'learning_rate': 0.022448982251243817, 'max_depth': 5, 'num_leaves': 27, 'subsample': 0.8264993906337597, 'colsample_bytree': 0.9169465091459863}. Best is trial 0 with value: 0.7711328349626222.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:54,400] Trial 1 finished with value: 0.777458309373203 and parameters: {'n_estimators': 50, 'l

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000604 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000789 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] N

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:54,562] Trial 5 finished with value: 0.7843588269120184 and parameters: {'n_estimators': 100, 'learning_rate': 0.019706101631946568, 'max_depth': 6, 'num_leaves': 28, 'subsample': 0.8056946280473402, 'colsample_bytree': 0.96589208037729}. Best is trial 3 with value: 0.7901092581943646.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:54,599] Trial 6 finished with value: 0.7619321449108684 and parameters: {'n_estimators': 50, 'learning_rate': 0.010981952475587212, 'max_depth': 5, 'num_leaves': 30, 'subsample': 0.8105578520781581, 'colsample

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000509 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000384 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] 

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:54,789] Trial 10 finished with value: 0.7964347326049454 and parameters: {'n_estimators': 100, 'learning_rate': 0.08745487427496021, 'max_depth': 6, 'num_leaves': 24, 'subsample': 0.8956896065207628, 'colsample_bytree': 0.8715185209903794}. Best is trial 10 with value: 0.7964347326049454.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:54,858] Trial 11 finished with value: 0.8027602070155262 and parameters: {'n_estimators': 100, 'learning_rate': 0.09415797492306505, 'max_depth': 6, 'num_leaves': 24, 'subsample': 0.8932413214875471, 'colsa

...

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:56,936] Trial 43 finished with value: 0.7981598619896493 and parameters: {'n_estimators': 100, 'learning_rate': 0.07298002337200517, 'max_depth': 6, 'num_leaves': 24, 'subsample': 0.9145061858150041, 'colsample_bytree': 0.865338588122353}. Best is trial 19 with value: 0.80448533640023.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:57,006] Trial 44 finished with value: 0.8021851638872916 and parameters: {'n_estimators': 100, 'learning_rate': 0.08685899199410536, 'max_depth': 6, 'num_leaves': 24, 'subsample': 0.8730784688926831, 'colsampl

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000487 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:57,109] Trial 46 finished with value: 0.7929844738355377 and parameters: {'n_estimators': 50, 'learning_rate': 0.08051772031149104, 'max_depth': 6, 'num_leaves': 23, 'subsample': 0.846254153250076, 'colsample_bytree': 0.9373922162246197}. Best is trial 19 with value: 0.80448533640023.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:08:57,180] Trial 47 finished with value: 0.7998849913743531 and parameters: {'n_estimators': 100, 'learning_rate': 0.09002572829673278, 'max_depth': 6, 'num_leaves': 24, 'subsample': 0.8552841324882084, 'colsample

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000321 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

[I 2025-04-27 15:08:57,546] Trial 0 finished with value: 0.7941345600920069 and parameters: {'n_estimators': 100, 'learning_rate': 0.024427861170439723, 'depth': 6, 'l2_leaf_reg': 2, 'border_count': 32}. Best is trial 0 with value: 0.7941345600920069.
[I 2025-04-27 15:08:57,741] Trial 1 finished with value: 0.78205865439908 and parameters: {'n_estimators': 100, 'learning_rate': 0.018693671480061794, 'depth': 4, 'l2_leaf_reg': 1, 'border_count': 64}. Best is trial 0 with value: 0.7941345600920069.
[I 2025-04-27 15:08:57,879] Trial 2 finished with value: 0.79700977573318 and parameters: {'n_estimators': 50, 'learning_rate': 0.07625069966337364, 'depth': 5, 'l2_leaf_reg': 2, 'border_count': 64}. Best is trial 2 with value: 0.79700977573318.
[I 2025-04-27 15:08:58,017] Trial 3 finished with value: 0.7809085681426107 and parameters: {'n_estimators': 50, 'learning_rate': 0.02646468169993639, 'depth': 5, 'l2_leaf_reg': 1, 'border_count': 64}. Best is trial 2 with value: 0.79700977573318.
[I 2

In [18]:
print("LGBM 最佳得分:", lgbm_best_value)
print("CatBoost 最佳得分:", catboost_best_value)
# 导出 LGBM 和 CatBoost 的最优参数到 JSON 文件
with open('lgbm_best_params.json', 'w') as f:
    json.dump(lgbm_best_params, f)

with open('catboost_best_params.json', 'w') as f:
    json.dump(catboost_best_params, f)

LGBM 最佳得分: 0.80448533640023
CatBoost 最佳得分: 0.8039102932719954


# 训练并保存模型

In [19]:

# 导入最优参数
with open('lgbm_best_params.json', 'r') as f:
    lgbm_best_params = json.load(f)

with open('catboost_best_params.json', 'r') as f:
    catboost_best_params = json.load(f)

# 使用最优参数重新实例化模型并训练
lgbm_model = LGBMClassifier(**lgbm_best_params, random_state=0)
catboost_model = CatBoostClassifier(**catboost_best_params, verbose=False, random_state=0)

lgbm_model.fit(X_train, y_train)
catboost_model.fit(X_train, y_train)

joblib.dump(lgbm_model, 'lgbm_best_model.joblib')
joblib.dump(catboost_model, 'catboost_best_model.joblib')


[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000536 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

['catboost_best_model.joblib']

# 模型融合与提交文件

In [ ]:
# 加载训练好的基模型
lgbm_model = joblib.load('lgbm_best_model.joblib')
catboost_model = joblib.load('catboost_best_model.joblib')

# 检查模型加载是否正确
print(lgbm_model.get_params())
print(catboost_model.get_params())

# 定义生成元特征的函数
def generate_meta_features(model, X_train, y_train, X_valid, y_valid, X_test, n_splits=5):
    print("Entering generate_meta_features function")  # 调试信息
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    meta_train = np.zeros((X_train.shape[0],))
    meta_valid = np.zeros((X_valid.shape[0],))
    meta_test = np.zeros((X_test.shape[0],))
    
    for train_index, val_index in kf.split(X_train):
        print("Inside KFold loop")  # 调试信息
        X_tr, X_val = X_train[train_index], X_train[val_index]
        y_tr = y_train[train_index]  # 获取当前折的训练目标变量
        
        # 确保 X_tr 和 y_tr 的维度正确
        print(f"X_tr shape: {X_tr.shape}, y_tr shape: {y_tr.shape}")
        
        model.fit(X_tr, y_tr)  # 传递 y_tr 作为目标变量
        meta_train[val_index] = model.predict_proba(X_val)[:, 1]
        print("Completed a fold")  # 调试信息
    
    # 使用整个训练集重新训练模型以生成验证集和测试集的元特征
    print("Refitting model on entire training set")  # 调试信息
    model.fit(X_train, y_train)
    meta_valid = model.predict_proba(X_valid)[:, 1]
    meta_test = model.predict_proba(X_test)[:, 1]
    print("Exiting generate_meta_features function")  # 调试信息
    
    return meta_train, meta_valid, meta_test

# 为 LGBM 和 CatBoost 生成元特征
print("Generating meta features for LGBM")  # 调试信息
lgbm_meta_train, lgbm_meta_valid, lgbm_meta_test = generate_meta_features(lgbm_model, X_train, y_train, X_valid, y_valid, X_test)
print("Generating meta features for CatBoost")  # 调试信息
catboost_meta_train, catboost_meta_valid, catboost_meta_test = generate_meta_features(catboost_model, X_train, y_train, X_valid, y_valid, X_test)

# 构建元特征矩阵
X_train_meta = np.column_stack((lgbm_meta_train, catboost_meta_train))
X_valid_meta = np.column_stack((lgbm_meta_valid, catboost_meta_valid))
X_test_meta = np.column_stack((lgbm_meta_test, catboost_meta_test))

# 定义元模型的目标函数
def meta_objective(trial):
    params = {
        'C': trial.suggest_float('C', 0.01, 10.0, log=True),
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']),
        'random_state': 0
    }
    model = LogisticRegression(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    accuracy = accuracy_score(y_valid, y_pred)
    print(f"Meta model accuracy: {accuracy:.4f}")  # 调试信息
    return accuracy

# 创建 Optuna 研究对象并进行优化
print("Starting Optuna study for meta model")  # 调试信息
meta_study = optuna.create_study(direction='maximize')
meta_study.optimize(meta_objective, n_trials=50)

# 获取最优参数
meta_best_params = meta_study.best_params
meta_best_value = meta_study.best_value

print(f"元模型最优参数: {meta_best_params}")
print(f"元模型最优准确率: {meta_best_value:.4f}")

# 使用最优参数训练最终的元模型
best_meta_model = LogisticRegression(**meta_best_params, random_state=0)
best_meta_model.fit(X_train_meta, y_train)

# 保存元模型
joblib.dump(best_meta_model, 'best_meta_model.joblib')


{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.8281387666499853, 'importance_type': 'split', 'learning_rate': 0.07959715449777605, 'max_depth': 6, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 23, 'objective': None, 'random_state': 0, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.9335691438865769, 'subsample_for_bin': 200000, 'subsample_freq': 0}
{'learning_rate': 0.08082568703487038, 'depth': 6, 'l2_leaf_reg': 2, 'border_count': 32, 'verbose': False, 'n_estimators': 100, 'random_state': 0}
Generating meta features for LGBM
Entering generate_meta_features function
Inside KFold loop
X_tr shape: (5563, 39), y_tr shape: (5563,)
[LightGBM] [Info] Number of positive: 2802, number of negative: 2761
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000255 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site

[LightGBM] [Info] Number of positive: 2815, number of negative: 2749
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000252 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2458
[LightGBM] [Info] Number of data points in the train set: 5564, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.505931 -> initscore=0.023725
[LightGBM] [Info] Start training from score 0.023725
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Completed a fold
Inside KFold loop
X_tr shape: (5563, 39), y_tr shape: (5563,)
Completed a fold
Inside KFold loop
X_tr shape: (5563, 39), y_tr shape: (5563,)
Completed a fold
Inside KFold loop
X_tr shape: (5563, 39), y_tr shape: (5563,)
Completed a fold
Inside KFold loop
X_tr shape: (5564, 39), y_tr shape: (5564,)
Completed a fold
Refitting model on entire training set


...

Exiting generate_meta_features function
Starting Optuna study for meta model
Meta model accuracy: 0.7999
Meta model accuracy: 0.8016
Meta model accuracy: 0.8005
Meta model accuracy: 0.7999
Meta model accuracy: 0.8005
Meta model accuracy: 0.7993
Meta model accuracy: 0.7999
Meta model accuracy: 0.7993
Meta model accuracy: 0.7993
Meta model accuracy: 0.7999
Meta model accuracy: 0.8039
Meta model accuracy: 0.8028
Meta model accuracy: 0.8045
Meta model accuracy: 0.8005
Meta model accuracy: 0.8039
Meta model accuracy: 0.7987
Meta model accuracy: 0.8010
Meta model accuracy: 0.7982
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.7993
Meta model accuracy: 0.8010
Meta model accuracy: 0.8033


[I 2025-04-27 15:09:09,451] Trial 23 finished with value: 0.8010350776308223 and parameters: {'C': 0.036494501273863326, 'solver': 'liblinear'}. Best is trial 12 with value: 0.80448533640023.
[I 2025-04-27 15:09:09,461] Trial 24 finished with value: 0.8033352501437608 and parameters: {'C': 0.012725705360074452, 'solver': 'liblinear'}. Best is trial 12 with value: 0.80448533640023.
[I 2025-04-27 15:09:09,469] Trial 25 finished with value: 0.7998849913743531 and parameters: {'C': 0.05455599496636119, 'solver': 'newton-cg'}. Best is trial 12 with value: 0.80448533640023.
[I 2025-04-27 15:09:09,478] Trial 26 finished with value: 0.7993099482461185 and parameters: {'C': 0.10639787912607557, 'solver': 'lbfgs'}. Best is trial 12 with value: 0.80448533640023.
[I 2025-04-27 15:09:09,485] Trial 27 finished with value: 0.8039102932719954 and parameters: {'C': 0.010283503557028374, 'solver': 'liblinear'}. Best is trial 12 with value: 0.80448533640023.
[I 2025-04-27 15:09:09,492] Trial 28 finished 

Meta model accuracy: 0.8010
Meta model accuracy: 0.8033
Meta model accuracy: 0.7999
Meta model accuracy: 0.7993
Meta model accuracy: 0.8039
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8016
Meta model accuracy: 0.8039
Meta model accuracy: 0.8010
Meta model accuracy: 0.8016
Meta model accuracy: 0.8005
Meta model accuracy: 0.8010
Meta model accuracy: 0.8016
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8010
Meta model accuracy: 0.7999
Meta model accuracy: 0.8045
Meta model accuracy: 0.8039
Meta model accuracy: 0.8045
Meta model accuracy: 0.8005
Meta model accuracy: 0.7999
Meta model accuracy: 0.8016
Meta model accuracy: 0.8039
Meta model accuracy: 0.8010


[I 2025-04-27 15:09:09,654] Trial 49 finished with value: 0.8004600345025877 and parameters: {'C': 0.062475858327271185, 'solver': 'liblinear'}. Best is trial 12 with value: 0.80448533640023.


Meta model accuracy: 0.8005
元模型最优参数: {'C': 0.01030273167520248, 'solver': 'liblinear'}
元模型最优准确率: 0.8045


In [ ]:

# 加载元模型
best_meta_model = joblib.load('best_meta_model.joblib')

# 生成预测结果
prob = best_meta_model.predict_proba(X_test_meta)[:, 1]

# 加载测试数据
test_data = pd.read_csv('test.csv')

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Transported': prob > 0.5
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)

In [21]:
# # 加载模型
# lgbm_loaded_model = joblib.load('lgbm_best_model.joblib')
# catboost_loaded_model = joblib.load('catboost_best_model.joblib')

# # 对测试数据进行预测，获取概率值
# lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]
# catboost_prob = catboost_model.predict_proba(X_test)[:, 1]

# # 简单平均融合
# ensemble_prob = (lgbm_prob + catboost_prob) / 2

# # 根据阈值生成最终预测
# threshold = 0.5
# ensemble_pred = ensemble_prob > threshold

# # 加载测试数据
# test_data = pd.read_csv('test.csv')  # 确保文件路径正确

# # 创建提交文件
# submission = pd.DataFrame({
#     'PassengerId': test_data['PassengerId'],  # 确保这里使用正确的列名
#     'Transported': ensemble_pred
# })

# # 保存为 CSV 文件
# submission.to_csv('submission.csv', index=False)
